In [3]:
from faker import Faker
import pandas as pd
import uuid
import random
import numpy as np

from pathlib import Path
from datetime import datetime, timedelta


# =========================================================
# Configuration
# =========================================================

fake = Faker()

Faker.seed(42)
random.seed(42)
np.random.seed(42)

PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)


# =========================================================
# Scale Configuration
# =========================================================

NUM_COMPANIES = 1_000
NUM_CUSTOMERS = 10_000
NUM_CONTACTS = 15_000

NUM_AGENTS = 200
NUM_TICKETS = 250_000

NUM_TICKET_EVENTS = 1_000_000
NUM_CSAT = 150_000

NUM_SUBSCRIPTIONS = 2_000
NUM_INVOICES = 100_000
NUM_PAYMENTS = 100_000

NUM_SESSIONS = 500_000
NUM_FEATURE_USAGE = 1_000_000
NUM_PRODUCT_EVENTS = 2_000_000


# =========================================================
# Helper Functions
# =========================================================

def random_uuid():
    return str(uuid.uuid4())


def random_datetime(start_years_ago=3, end_years_ago=0):
    return fake.date_time_between(
        start_date=f"-{start_years_ago}y",
        end_date="now"
    )


def save_parquet(df, filename):
    output_path = OUTPUT_DIR / filename

    df.to_parquet(
        output_path,
        index=False
    )

    print(
        f"Created {filename}: "
        f"{len(df):,} rows"
    )


# =========================================================
# Companies
# =========================================================

def generate_companies(num_companies=NUM_COMPANIES):

    industries = [
        "Technology",
        "FinTech",
        "Healthcare",
        "Retail",
        "Logistics",
        "Education",
        "Manufacturing",
        "Travel"
    ]

    company_sizes = [
        "SMB",
        "Mid-Market",
        "Enterprise"
    ]

    companies = []

    for _ in range(num_companies):

        size = random.choice(company_sizes)

        if size == "SMB":
            revenue = random.uniform(100_000, 5_000_000)

        elif size == "Mid-Market":
            revenue = random.uniform(5_000_000, 50_000_000)

        else:
            revenue = random.uniform(
                50_000_000,
                500_000_000
            )

        companies.append({
            "company_id": random_uuid(),
            "company_name": fake.company(),
            "industry": random.choice(industries),
            "company_size": size,
            "country": fake.country(),
            "city": fake.city(),
            "annual_revenue": round(revenue, 2),
            "created_at": random_datetime(5)
        })

    return pd.DataFrame(companies)


# =========================================================
# Customers
# =========================================================

def generate_customers(
    companies,
    num_customers=NUM_CUSTOMERS
):

    customers = []

    company_ids = companies["company_id"].tolist()

    for _ in range(num_customers):

        company_id = random.choice(company_ids)

        customers.append({
            "customer_id": random_uuid(),
            "company_id": company_id,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.email(),
            "job_title": fake.job(),
            "country": fake.country(),
            "created_at": random_datetime(3),
            "status": random.choice([
                "active",
                "active",
                "active",
                "inactive"
            ])
        })

    return pd.DataFrame(customers)


# =========================================================
# Contacts
# =========================================================

def generate_contacts(
    customers,
    num_contacts=NUM_CONTACTS
):

    contacts = []

    customer_ids = customers["customer_id"].tolist()

    contact_types = [
        "Primary",
        "Billing",
        "Technical",
        "Executive",
        "Support"
    ]

    for _ in range(num_contacts):

        contacts.append({
            "contact_id": random_uuid(),
            "customer_id": random.choice(customer_ids),
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.email(),
            "phone": fake.phone_number(),
            "contact_type": random.choice(contact_types),
            "created_at": random_datetime(3)
        })

    return pd.DataFrame(contacts)


# =========================================================
# Agents
# =========================================================

def generate_agents(num_agents=NUM_AGENTS):

    teams = [
        "Tier 1 Support",
        "Tier 2 Support",
        "Technical Support",
        "Billing",
        "Customer Success"
    ]

    agents = []

    for _ in range(num_agents):

        agents.append({
            "agent_id": random_uuid(),
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.email(),
            "team": random.choice(teams),
            "country": fake.country(),
            "hire_date": fake.date_between(
                start_date="-5y",
                end_date="today"
            ),
            "status": random.choice([
                "active",
                "active",
                "active",
                "inactive"
            ])
        })

    return pd.DataFrame(agents)


# =========================================================
# Tickets
# =========================================================

def generate_tickets(
    customers,
    agents,
    num_tickets=NUM_TICKETS
):

    priorities = [
        "low",
        "medium",
        "high",
        "urgent"
    ]

    channels = [
        "email",
        "chat",
        "phone",
        "web"
    ]

    categories = [
        "Technical Issue",
        "Billing",
        "Account",
        "Feature Request",
        "Bug",
        "Integration"
    ]

    statuses = [
        "open",
        "pending",
        "resolved",
        "closed"
    ]

    tickets = []

    customer_ids = customers["customer_id"].tolist()
    agent_ids = agents["agent_id"].tolist()

    for _ in range(num_tickets):

        created_at = random_datetime(1)

        priority = random.choice(priorities)
        status = random.choice(statuses)

        first_response_at = (
            created_at
            + timedelta(
                minutes=random.randint(1, 240)
            )
        )

        if status in ["resolved", "closed"]:

            resolution_minutes = random.randint(
                30,
                7 * 24 * 60
            )

            resolved_at = (
                created_at
                + timedelta(
                    minutes=resolution_minutes
                )
            )

        else:

            resolved_at = None

        tickets.append({
            "ticket_id": random_uuid(),
            "customer_id": random.choice(customer_ids),
            "agent_id": random.choice(agent_ids),
            "subject": fake.sentence(nb_words=8),
            "category": random.choice(categories),
            "priority": priority,
            "channel": random.choice(channels),
            "status": status,
            "created_at": created_at,
            "first_response_at": first_response_at,
            "resolved_at": resolved_at
        })

    return pd.DataFrame(tickets)


# =========================================================
# Ticket Events
# =========================================================

def generate_ticket_events(
    tickets,
    num_events=NUM_TICKET_EVENTS
):

    event_types = [
        "created",
        "assigned",
        "customer_reply",
        "agent_reply",
        "status_changed",
        "priority_changed",
        "escalated",
        "resolved",
        "reopened"
    ]

    events = []

    ticket_ids = tickets["ticket_id"].tolist()

    for _ in range(num_events):

        ticket_id = random.choice(ticket_ids)

        event_time = random_datetime(1)

        events.append({
            "ticket_event_id": random_uuid(),
            "ticket_id": ticket_id,
            "event_type": random.choice(event_types),
            "event_timestamp": event_time,
            "actor_type": random.choice([
                "customer",
                "agent",
                "system"
            ])
        })

    return pd.DataFrame(events)


# =========================================================
# Customer Satisfaction
# =========================================================

def generate_customer_satisfaction(
    tickets,
    num_records=NUM_CSAT
):

    satisfaction = []

    ticket_ids = tickets["ticket_id"].tolist()

    for _ in range(num_records):

        satisfaction.append({
            "csat_id": random_uuid(),
            "ticket_id": random.choice(ticket_ids),
            "rating": random.choices(
                [1, 2, 3, 4, 5],
                weights=[
                    5,
                    8,
                    15,
                    30,
                    42
                ]
            )[0],
            "feedback": fake.sentence(
                nb_words=12
            ),
            "submitted_at": random_datetime(1)
        })

    return pd.DataFrame(satisfaction)


# =========================================================
# Plans
# =========================================================

def generate_plans():

    plans = [
        {
            "plan_id": "PLAN-001",
            "plan_name": "Starter",
            "monthly_price": 99,
            "max_users": 10
        },
        {
            "plan_id": "PLAN-002",
            "plan_name": "Professional",
            "monthly_price": 499,
            "max_users": 50
        },
        {
            "plan_id": "PLAN-003",
            "plan_name": "Business",
            "monthly_price": 999,
            "max_users": 200
        },
        {
            "plan_id": "PLAN-004",
            "plan_name": "Enterprise",
            "monthly_price": 4999,
            "max_users": 1000
        }
    ]

    return pd.DataFrame(plans)


# =========================================================
# Subscriptions
# =========================================================

def generate_subscriptions(
    companies,
    plans,
    num_subscriptions=NUM_SUBSCRIPTIONS
):

    subscriptions = []

    company_ids = companies["company_id"].tolist()

    for _ in range(num_subscriptions):

        plan = plans.sample(1).iloc[0]

        start_date = fake.date_between(
            start_date="-3y",
            end_date="today"
        )

        status = random.choice([
            "active",
            "active",
            "active",
            "cancelled",
            "paused"
        ])

        subscriptions.append({
            "subscription_id": random_uuid(),
            "company_id": random.choice(company_ids),
            "plan_id": plan["plan_id"],
            "start_date": start_date,
            "status": status,
            "monthly_price": plan["monthly_price"]
        })

    return pd.DataFrame(subscriptions)


# =========================================================
# Invoices
# =========================================================

def generate_invoices(
    subscriptions,
    num_invoices=NUM_INVOICES
):

    invoices = []

    subscription_ids = (
        subscriptions["subscription_id"]
        .tolist()
    )

    invoice_statuses = [
        "paid",
        "paid",
        "paid",
        "open",
        "overdue"
    ]

    for _ in range(num_invoices):

        invoice_date = fake.date_between(
            start_date="-2y",
            end_date="today"
        )

        due_date = (
            invoice_date
            + timedelta(days=30)
        )

        subscription = subscriptions.sample(
            1
        ).iloc[0]

        invoices.append({
            "invoice_id": random_uuid(),
            "subscription_id": subscription[
                "subscription_id"
            ],
            "invoice_date": invoice_date,
            "due_date": due_date,
            "amount": subscription[
                "monthly_price"
            ],
            "status": random.choice(
                invoice_statuses
            )
        })

    return pd.DataFrame(invoices)


# =========================================================
# Payments
# =========================================================

def generate_payments(
    invoices,
    num_payments=NUM_PAYMENTS
):

    payments = []

    invoice_ids = invoices["invoice_id"].tolist()

    payment_methods = [
        "credit_card",
        "bank_transfer",
        "ach",
        "paypal"
    ]

    for _ in range(num_payments):

        invoice = invoices.sample(1).iloc[0]

        payment_date = (
            invoice["invoice_date"]
            + timedelta(
                days=random.randint(1, 35)
            )
        )

        payments.append({
            "payment_id": random_uuid(),
            "invoice_id": invoice["invoice_id"],
            "payment_date": payment_date,
            "amount": invoice["amount"],
            "payment_method": random.choice(
                payment_methods
            ),
            "status": random.choice([
                "successful",
                "successful",
                "successful",
                "failed"
            ])
        })

    return pd.DataFrame(payments)


# =========================================================
# Sessions
# =========================================================

def generate_sessions(
    customers,
    num_sessions=NUM_SESSIONS
):

    sessions = []

    customer_ids = customers["customer_id"].tolist()

    devices = [
        "desktop",
        "mobile",
        "tablet"
    ]

    for _ in range(num_sessions):

        session_start = random_datetime(1)

        sessions.append({
            "session_id": random_uuid(),
            "customer_id": random.choice(
                customer_ids
            ),
            "session_start": session_start,
            "session_end": (
                session_start
                + timedelta(
                    minutes=random.randint(
                        2,
                        180
                    )
                )
            ),
            "device": random.choice(devices),
            "country": fake.country()
        })

    return pd.DataFrame(sessions)


# =========================================================
# Feature Usage
# =========================================================

def generate_feature_usage(
    customers,
    num_records=NUM_FEATURE_USAGE
):

    features = [
        "Ticket Management",
        "Knowledge Base",
        "Live Chat",
        "Automation",
        "Reporting",
        "Analytics",
        "Integrations",
        "Customer Portal"
    ]

    usage = []

    customer_ids = customers["customer_id"].tolist()

    for _ in range(num_records):

        usage.append({
            "feature_usage_id": random_uuid(),
            "customer_id": random.choice(
                customer_ids
            ),
            "feature_name": random.choice(
                features
            ),
            "usage_count": random.randint(
                1,
                100
            ),
            "usage_date": fake.date_between(
                start_date="-1y",
                end_date="today"
            )
        })

    return pd.DataFrame(usage)


# =========================================================
# Product Events
# =========================================================

def generate_product_events(
    customers,
    num_events=NUM_PRODUCT_EVENTS
):

    event_names = [
        "login",
        "ticket_created",
        "ticket_viewed",
        "report_created",
        "dashboard_viewed",
        "feature_enabled",
        "integration_connected",
        "knowledge_article_viewed",
        "chat_started",
        "settings_updated"
    ]

    events = []

    customer_ids = customers["customer_id"].tolist()

    for _ in range(num_events):

        events.append({
            "event_id": random_uuid(),
            "customer_id": random.choice(
                customer_ids
            ),
            "event_name": random.choice(
                event_names
            ),
            "event_timestamp": random_datetime(1),
            "source": random.choice([
                "web",
                "mobile",
                "api"
            ])
        })

    return pd.DataFrame(events)


# =========================================================
# Main
# =========================================================

def main():

    print(
        "\nGenerating SupportIQ SaaS "
        "customer analytics data...\n"
    )

    # -------------------------
    # CRM
    # -------------------------

    companies = generate_companies()

    customers = generate_customers(
        companies
    )

    contacts = generate_contacts(
        customers
    )

    # -------------------------
    # Support
    # -------------------------

    agents = generate_agents()

    tickets = generate_tickets(
        customers,
        agents
    )

    ticket_events = generate_ticket_events(
        tickets
    )

    customer_satisfaction = (
        generate_customer_satisfaction(
            tickets
        )
    )

    # -------------------------
    # Billing
    # -------------------------

    plans = generate_plans()

    subscriptions = generate_subscriptions(
        companies,
        plans
    )

    invoices = generate_invoices(
        subscriptions
    )

    payments = generate_payments(
        invoices
    )

    # -------------------------
    # Product
    # -------------------------

    sessions = generate_sessions(
        customers
    )

    feature_usage = generate_feature_usage(
        customers
    )

    product_events = generate_product_events(
        customers
    )

    # -------------------------
    # Save CRM
    # -------------------------

    save_parquet(
        companies,
        "companies.parquet"
    )

    save_parquet(
        customers,
        "customers.parquet"
    )

    save_parquet(
        contacts,
        "contacts.parquet"
    )

    # -------------------------
    # Save Support
    # -------------------------

    save_parquet(
        agents,
        "agents.parquet"
    )

    save_parquet(
        tickets,
        "tickets.parquet"
    )

    save_parquet(
        ticket_events,
        "ticket_events.parquet"
    )

    save_parquet(
        customer_satisfaction,
        "customer_satisfaction.parquet"
    )

    # -------------------------
    # Save Billing
    # -------------------------

    save_parquet(
        plans,
        "plans.parquet"
    )

    save_parquet(
        subscriptions,
        "subscriptions.parquet"
    )

    save_parquet(
        invoices,
        "invoices.parquet"
    )

    save_parquet(
        payments,
        "payments.parquet"
    )

    # -------------------------
    # Save Product
    # -------------------------

    save_parquet(
        sessions,
        "sessions.parquet"
    )

    save_parquet(
        feature_usage,
        "feature_usage.parquet"
    )

    save_parquet(
        product_events,
        "product_events.parquet"
    )

    print(
        "\nData generation completed."
    )

    print(
        f"Output directory: {OUTPUT_DIR}"
    )


# =========================================================
# Run
# =========================================================

main()

Project root: /Users/rishimukherjee/customer-analytics-platform
Output directory: /Users/rishimukherjee/customer-analytics-platform/data/raw

Generating SupportIQ SaaS customer analytics data...

Created companies.parquet: 1,000 rows
Created customers.parquet: 10,000 rows
Created contacts.parquet: 15,000 rows
Created agents.parquet: 200 rows
Created tickets.parquet: 250,000 rows
Created ticket_events.parquet: 1,000,000 rows
Created customer_satisfaction.parquet: 150,000 rows
Created plans.parquet: 4 rows
Created subscriptions.parquet: 2,000 rows
Created invoices.parquet: 100,000 rows
Created payments.parquet: 100,000 rows
Created sessions.parquet: 500,000 rows
Created feature_usage.parquet: 1,000,000 rows
Created product_events.parquet: 2,000,000 rows

Data generation completed.
Output directory: /Users/rishimukherjee/customer-analytics-platform/data/raw
